# Multi-Agent System Design Patterns

One agent with twenty tools becomes unreliable: the context fills with
irrelevant tool chatter and every mistake compounds. Splitting the work
across **specialist agents** restores focus — at the price of coordination.
This session builds the two patterns that cover most real systems: the
**supervisor** (routing) and the **writer–critic** (review loop).

## Agents as roles

An agent here is a name, a role, and a policy over messages. Keeping the
interface this small is what lets the same coordination code run scripted
policies today and real models tomorrow.

In [1]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

DOC_VECTORS = {name: embed(doc) for name, doc in CORPUS.items()}

class Agent:
    def __init__(self, name, role, policy):
        self.name, self.role, self.policy = name, role, policy

    def act(self, message):
        reply = self.policy(message)
        print(f"[{self.name}] {reply[:76]}")
        return reply

def kb_answer(topic_filter):
    # A specialist policy: answer only from this specialist's documents.
    def policy(message):
        candidates = {n: v for n, v in DOC_VECTORS.items()
                      if n in topic_filter}
        name = max(candidates,
                   key=lambda n: cosine(embed(message), candidates[n]))
        return f"{CORPUS[name][:70]}... [source: {name}]"
    return policy

support = Agent(
    "support",
    "battery care, charging, riding range, display error codes",
    kb_answer({"battery-care", "range", "error-codes"}))
accounts = Agent(
    "accounts",
    "warranty coverage, covered replacement policy, service bookings",
    kb_answer({"warranty", "first-service"}))
print("specialists ready:", support.name + ",", accounts.name)

specialists ready: support, accounts


## Pattern 1 — Supervisor

A supervisor never answers; it **routes**. Scoped workers mean scoped
context and scoped tools — the whole reason to go multi-agent. In
production the routing policy is a cheap, fast model; here it is cosine
similarity against the role descriptions.

In [2]:
class Supervisor:
    def __init__(self, workers):
        self.workers = workers

    def route(self, ticket):
        scored = [(cosine(embed(ticket), embed(w.role)), w)
                  for w in self.workers]
        score, worker = max(scored, key=lambda x: x[0])
        print(f"[supervisor] -> {worker.name} (fit {score:.2f})")
        return worker.act(ticket)

desk = Supervisor([support, accounts])
for ticket in [
    "my display shows E01 after a rainy ride",
    "is the motor replacement covered in year two",
]:
    print(f"ticket: {ticket!r}")
    desk.route(ticket)
    print()

ticket: 'my display shows E01 after a rainy ride'
[supervisor] -> support (fit 0.22)
[support] Atlas display error codes. E01 means a motor sensor fault: restart the... [s

ticket: 'is the motor replacement covered in year two'
[supervisor] -> accounts (fit 0.39)
[accounts] Atlas warranty policy. The frame is covered for 5 years. The battery a... [s



## Pattern 2 — Writer and critic

The second reliable pattern is adversarial: one agent drafts, another
checks the draft against an explicit rubric, and the writer revises once.
The rubric matters more than the critic — a critic without concrete checks
just says "looks good".

In [3]:
def writer_policy(message):
    # First draft is deliberately sloppy: no source citation.
    name = max(DOC_VECTORS, key=lambda n: cosine(embed(message), DOC_VECTORS[n]))
    return f"Customer reply: {CORPUS[name][:80]}..."

def critic_policy(draft):
    problems = []
    if "[source:" not in draft:
        problems.append("cite the knowledge-base source")
    if len(draft) > 300:
        problems.append("shorten to under 300 chars")
    return "APPROVE" if not problems else "REVISE: " + "; ".join(problems)

writer = Agent("writer", "drafts customer replies", writer_policy)
critic = Agent("critic", "reviews drafts against the rubric", critic_policy)

question = "how should I store the battery over winter"
draft = writer.act(question)
verdict = critic.act(draft)
if verdict.startswith("REVISE"):
    name = max(DOC_VECTORS, key=lambda n: cosine(embed(question), DOC_VECTORS[n]))
    draft = draft + f" [source: {name}]"
    print(f"[writer] revised with citation [source: {name}]")
    verdict = critic.act(draft)
print(f"\nfinal verdict: {verdict}")

[writer] Customer reply: Atlas S2 battery care. Charge the battery to 80 percent for 
[critic] REVISE: cite the knowledge-base source
[writer] revised with citation [source: battery-care]
[critic] APPROVE

final verdict: APPROVE


## Failure modes — and what buys them back

| Failure | Mitigation |
|---|---|
| Two agents loop forever politely deferring | Step budgets; a supervisor with authority to stop |
| Workers drift out of sync on shared facts | One shared state/store, not N private memories |
| Costs multiply per agent per turn | Route with a cheap model; reserve the big model for the work |

**Takeaways**
- Go multi-agent for *context isolation and tool scoping*, not for theater.
- Supervisor = routing; writer–critic = review. Most systems are these two.
- Budgets and shared state are not optional; they are the coordination tax.

Tomorrow: memory that survives the session, and evaluation that catches the
failures you just met — before your users do.